1) Positiv/Negativ-Klassifikator 

Überwachte Klassifikation: Snippet + Deployment-KPIs → Label `1` (negativ/mutiert) und `0` (positiv).

Dient als Proof of Concept, ob die Snippets anhand der kpi Metriken in positiv und negatibeispiele eingeteilt werden können. 


In [135]:
import numpy as np
import pandas as pd
import sklearn

import pathlib


from sklearn.model_selection import GroupShuffleSplit
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

In [136]:
ROOT = pathlib.Path("/Users/svenniederlohner/projects/Bachelorthesis_KI_gestuetztes_deployment")
KPI_CSV_DIR = ROOT / "export_kpis" / "csv_kpis"
KPI_CSV_CANDIDATES = sorted(KPI_CSV_DIR.glob("all_projects_kpis_[0-9]*.csv"))
if not KPI_CSV_CANDIDATES:
 raise FileNotFoundError(
 "Keine Zeitstempel-KPI-CSV in export_kpis/csv_kpis/ gefunden - zuerst KPIs aus dem Bucket ziehen."
)
KPI_CSV = KPI_CSV_CANDIDATES[-1]

df = pd.read_csv(KPI_CSV)

is_neg = df["variant"].str.endswith("_neg")
neg = is_neg.sum()
pos = (~is_neg).sum()
print("Zeile insgesammt", len(df))
print("Davon positive Datensaetze :", pos)
print("Negative Datensaetze :", neg)

if neg == 0 or pos == 0:
    raise ValueError(
        "Entweder keine positiven oder negativen datensaetze vorhanden"
    ) 

Zeile insgesammt 705
Davon positive Datensaetze : 600
Negative Datensaetze : 105


2) Label & Feature Matrix 

- label is_neg -> negativ Endung -> wird zu 1 (negativ)
- base_project : Projekt ohne _neg endung  -> 0 (positiv)
- pair-id : Pärchen Gruppen -> pos+neg Code bleiben zusammen
 

In [137]:
df["is_neg"] = df["variant"].str.endswith("_neg").astype(int)
df["project_id"] = df["variant"].str.replace("_neg","", regex=False)
df["pair_id"] = df["target_method"]

In [138]:
print("Spalten:", list(df.columns))
print()
print("Projekte:", sorted(df["project_id"].unique()))
print("Umgebungen:", sorted(df["env"].unique()))
print()
metric_cols = ["avg_latency_ms", "error_rate_percent", "p95_latency_ms", "requests_per_sec", "total_failures", "total_requests"]
print(df[metric_cols].describe().T)
print()
print("Fehlende Werte:")
print(df[metric_cols].isna().sum())
print()
print("Zeilen je Projekt und Label:")
print(df.groupby(["project_id", "is_neg"]).size().unstack(fill_value=0))

Spalten: ['variant', 'env', 'source', 'avg_latency_ms', 'error_rate_percent', 'p95_latency_ms', 'requests_per_sec', 'target_method', 'total_failures', 'total_requests', 'is_neg', 'project_id', 'pair_id']

Projekte: ['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13']
Umgebungen: ['extreme', 'high', 'low', 'medium', 'prod']

                    count         mean           std    min     25%      50%  \
avg_latency_ms      705.0  3322.228780   5687.238357   2.94   50.03   397.24   
error_rate_percent  705.0     2.982426     13.281872   0.00    0.00     0.00   
p95_latency_ms      705.0  7520.777305  13753.054053   3.00  180.00   650.00   
requests_per_sec    705.0    49.825447     52.343662   0.34    3.85    36.08   
total_failures      705.0    28.496454    216.644360   0.00    0.00     0.00   
total_requests      705.0  3927.205674   5394.171252  10.00  289.00  1254.00   

                        75%       max  
avg_latency_ms      4945.37  44660

In [139]:
RANDOM_STATE = 42
TARGET = "is_neg"

TRAINING_PROJECTS = ["P01", "P02", "P04", "P05", "P06", "P07", "P08", "P09", "P10", "P11"]
EVAL_PROJECTS = ["P12", "P13"]

NUM_FEATURES = [
    "avg_latency_ms",
    "error_rate_percent",
    "p95_latency_ms",
    "requests_per_sec",
]
LOG_FEATURES =["avg_latency_ms", "p95_latency_ms", "requests_per_sec"]
PLAIN_FEATURES = ["error_rate_percent"]
CAT_FEATURES = ["env"]

print("Trainings-Projekte (P01-P11 ohne P03):", TRAINING_PROJECTS)
print("Held-out Evaluations-Projekte:", EVAL_PROJECTS)
print("Numerische Features:", NUM_FEATURES)
print("Kategorische Features:", CAT_FEATURES)

Trainings-Projekte (P01-P11 ohne P03): ['P01', 'P02', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11']
Held-out Evaluations-Projekte: ['P12', 'P13']
Numerische Features: ['avg_latency_ms', 'error_rate_percent', 'p95_latency_ms', 'requests_per_sec']
Kategorische Features: ['env']


3) Training und Evaluation Datenset aufteilen 


In [140]:
train_val = df[df["project_id"].isin(TRAINING_PROJECTS)]
test_df = df[df["project_id"].isin(EVAL_PROJECTS)]

shufflesplit = GroupShuffleSplit (n_splits=1,test_size=0.25,random_state=RANDOM_STATE)
train_idx,val_idx = next(shufflesplit.split(train_val, groups= train_val["pair_id"]))

train_df = train_val.iloc[train_idx].copy()
val_df = train_val.iloc[val_idx].copy()

print("KPI-CSV", KPI_CSV.name)
print("Train-Projekte:", sorted(train_df["project_id"].unique()))
print("Val-Projekte:  ", sorted(val_df["project_id"].unique()))
print("Test (held-out):", sorted(test_df["project_id"].unique()))
print("n train / val / test:", len(train_df), len(val_df), len(test_df))
print("Klassenverteilung train (0 = positiv, 1 = negativ):")
print(train_df[TARGET].value_counts().sort_index())
print("Klassenverteilung val (0 = positiv, 1 = negativ):")
print(val_df[TARGET].value_counts().sort_index())

KPI-CSV all_projects_kpis_20260911_203947.csv
Train-Projekte: ['P01', 'P02', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11']
Val-Projekte:   ['P01', 'P02', 'P04', 'P05', 'P06', 'P07', 'P08', 'P10']
Test (held-out): ['P12', 'P13']
n train / val / test: 380 130 185
Klassenverteilung train (0 = positiv, 1 = negativ):
is_neg
0    300
1     80
Name: count, dtype: int64
Klassenverteilung val (0 = positiv, 1 = negativ):
is_neg
0    105
1     25
Name: count, dtype: int64


4) Preprocessing

- Fehlende Kpi werte ersetzen über den median
- Numerische Kpis standardisieren -> Wichtig für die Logistische Regression
- Die Umgebung (env) wird one-hot encodiert

Der Preprocessor wird ausschließlich auf die Trainingsdaten trainiert und dann auf die Validierungs und Evaluierungsdaten angewandt.

Untersuchungsschritt: Im endgültigen Modell (Abschnitt 11) entfällt die Umgebungs-Kodierung, weil sie messbar nichts beiträgt. Die Imputation, die Logarithmierung und die Skalierung bleiben.

In [141]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

log_pipeline = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy="median")), 
        ("log",FunctionTransformer (np.log1p, feature_names_out = "one-to-one")),
        ("scaler", StandardScaler()),
    ]
)

plain_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("onehot", OneHotEncoder(handle_unknown="ignore", drop=["low"])),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("log", log_pipeline, LOG_FEATURES),
        ("plain", plain_pipeline, PLAIN_FEATURES),
        ("cat", categorical_pipeline, CAT_FEATURES),
    ],
    sparse_threshold=0.0,
    verbose_feature_names_out=False,
)

X_train = preprocessor.fit_transform(
    train_df[LOG_FEATURES + PLAIN_FEATURES + CAT_FEATURES]
)
X_val = preprocessor.transform(val_df[LOG_FEATURES + PLAIN_FEATURES + CAT_FEATURES])
y_train = train_df[TARGET]
y_val = val_df[TARGET]

print("Erzeugte Feature-Spalten:", list(preprocessor.get_feature_names_out()))
print("X_train:", X_train.shape, "| X_val:", X_val.shape)
print(
    "Fehlende Werte train / val:",
    int(np.isnan(X_train).sum()),
    int(np.isnan(X_val).sum()),
)
print("Klassen train (0 / 1):", int((y_train == 0).sum()), int((y_train == 1).sum()))
print("Klassen val   (0 / 1):", int((y_val == 0).sum()), int((y_val == 1).sum()))

Erzeugte Feature-Spalten: ['avg_latency_ms', 'p95_latency_ms', 'requests_per_sec', 'error_rate_percent', 'env_extreme', 'env_high', 'env_medium', 'env_prod']
X_train: (380, 8) | X_val: (130, 8)
Fehlende Werte train / val: 0 0
Klassen train (0 / 1): 300 80
Klassen val   (0 / 1): 105 25


5. Modelle und Kennzahlen

- Logistische Regression in zwei Gewichtungen
- Dummy-Klassifikator dient als Vergleich
- Kennzahlen: PR-AUC, F1, Precision, Recall, Confusion Matrix

Untersuchungsschritt: Hier entsteht die erste Vergleichslinie überhaupt. Das endgültige Modell steht in Abschnitt 11.

In [142]:
modelle = {
    "LogReg (keine Gewichtung)": LogisticRegression(
        max_iter=1000, random_state=RANDOM_STATE
    ),
    "LogReg (balanced)": LogisticRegression(
        max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced"
    ),
    "Dummy (most_frequent)": DummyClassifier(strategy="most_frequent"),
}

ergebnisse = []

for name, klassifikator in modelle.items():
    klassifikator.fit(X_train, y_train)
    y_pred = klassifikator.predict(X_val)
    y_score = klassifikator.predict_proba(X_val)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()
    ergebnisse.append(
        {
            "Modell": name,
            "PR-AUC": round(average_precision_score(y_val, y_score), 3),
            "F1": round(f1_score(y_val, y_pred, zero_division=0), 3),
            "Precision": round(precision_score(y_val, y_pred, zero_division=0), 3),
            "Recall": round(recall_score(y_val, y_pred, zero_division=0), 3),
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "TP": tp,
        }
    )

ergebnis_df = pd.DataFrame(ergebnisse).set_index("Modell")

print("Zufalls-Baseline PR-AUC (Anteil der Negativen in val):", round(y_val.mean(), 3))
print()
print(ergebnis_df)

Zufalls-Baseline PR-AUC (Anteil der Negativen in val): 0.192

                           PR-AUC     F1  Precision  Recall   TN  FP  FN  TP
Modell                                                                      
LogReg (keine Gewichtung)   0.607  0.077      1.000    0.04  105   0  24   1
LogReg (balanced)           0.620  0.444      0.297    0.88   53  52   3  22
Dummy (most_frequent)       0.192  0.000      0.000    0.00  105   0  25   0


6. Entscheidungsgrenze und Fehleranalyse

- Der Standardwert 0,5 ist bei 19 % Negativanteil unbrauchbar (Recall 0,04)
-  Die Grenze wird aus ehrlichen Trainingsdaten bestimmt (Out-of-Fold über GroupKFold), nicht auf der Validierung
-  Recall-Ziel 0,8: eine übersehene Mutation (FN) ist teurer als ein Fehlalarm (FP)
-  Die Grenze wird anschließend nur einmal auf die Validierung angewandt (kein Nachjustieren)
-  Die Trennschärfe ändert sich dadurch nicht: die PR-AUC bleibt unverändert
-  Fehleranalyse: welche Mutationen übersehen wurden und wo die Fehlalarme sitzen (je Umgebungsstufe)

In [143]:
from sklearn.metrics import precision_recall_curve
from sklearn.model_selection import GroupKFold, cross_val_predict

logreg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)

scores_train = cross_val_predict(
    logreg,
    X_train,
    y_train,
    cv=GroupKFold(n_splits=5),
    groups=train_df["pair_id"],
    method="predict_proba",
)[:, 1]

precision, recall, schwellen = precision_recall_curve(y_train, scores_train)

ziel_recall = 0.8
geeignet = recall[:-1] >= ziel_recall
SCHWELLE = schwellen[geeignet].max()

logreg.fit(X_train, y_train)
y_score_val = logreg.predict_proba(X_val)[:, 1]
y_pred_val = (y_score_val >= SCHWELLE).astype(int)

tn, fp, fn, tp = confusion_matrix(y_val, y_pred_val).ravel()

print("PR-AUC train (Out-of-Fold):", round(average_precision_score(y_train, scores_train), 3))
print("Zufalls-Baseline train:", round(y_train.mean(), 3))
print("Gewählter Schwellenwert:", round(SCHWELLE, 3), "für Recall-Ziel", ziel_recall)
print()
print("PR-AUC val:", round(average_precision_score(y_val, y_score_val), 3))
print("F1:", round(f1_score(y_val, y_pred_val, zero_division=0), 3))
print("Precision:", round(precision_score(y_val, y_pred_val, zero_division=0), 3))
print("Recall:", round(recall_score(y_val, y_pred_val, zero_division=0), 3))
print()
print("Mutation erkannt (TP):", tp)
print("Mutation übersehen (FN):", fn)
print("Fehlalarm (FP):", fp)
print("korrekt freigegeben (TN):", tn)

PR-AUC train (Out-of-Fold): 0.367
Zufalls-Baseline train: 0.211
Gewählter Schwellenwert: 0.172 für Recall-Ziel 0.8

PR-AUC val: 0.607
F1: 0.431
Precision: 0.286
Recall: 0.88

Mutation erkannt (TP): 22
Mutation übersehen (FN): 3
Fehlalarm (FP): 55
korrekt freigegeben (TN): 50


6b) Fehleranalyse

- Die Kennzahlen verraten nur das etwas nicht passt. Eine genauere Fehleranalyse findet nun statt.

Was ich hier wissen will:

- welche Mutationen wurden übersehen?
- wie gut werden die Fehler je Umgebung erkannt? Greift das Modell unter Last besser?
- wo sitzen die Fehlalarme? Fallen sie vor allem in high und extreme?

Der letzte Punkt ist der wichtigste. Wenn die Fehlalarme fast nur in high und extreme hängen, dann reagiert das Modell einfach auf die Last und nicht auf die Mutation. Merkmale werden dann umgestellt auf lastunabhängigere(Verhältnis zum Positiv-Snippet derselben Umgebung). 

Nebenbei schaue ich mir noch an, welche Methoden überhaupt in der Validierung liegen. Das erklärt hoffentlich, warum die Out-of-Fold-Zahl (0,367) so viel schlechter aussieht als die Validierungszahl (0,607) 
- nämlich dann, wenn in der Validierung vor allem die gut messbaren Fälle gelandet sind.

In [144]:
fehler = val_df[["project_id", "target_method", "env", TARGET]].copy()
fehler["score"] = y_score_val
fehler["vorhersage"] = y_pred_val

print("Negativ-Methoden in der Validierung:")
print(sorted(val_df.loc[val_df[TARGET] == 1, "target_method"].unique()))
print()

uebersehen = fehler[(fehler[TARGET] == 1) & (fehler["vorhersage"] == 0)]
print("Übersehene Mutationen (FN):", len(uebersehen))
print(uebersehen[["target_method", "env", "score"]].sort_values("score", ascending=False))
print()

mutationen = fehler[fehler[TARGET] == 1].groupby("env").size().rename("mutationen")
erkannt = (
    fehler[(fehler[TARGET] == 1) & (fehler["vorhersage"] == 1)]
    .groupby("env")
    .size()
    .rename("erkannt")
)
trefferquote = mutationen.to_frame().join(erkannt).fillna(0).astype(int)
trefferquote["trefferquote"] = (trefferquote["erkannt"] / trefferquote["mutationen"]).round(3)
print("Erkennungsquote je Umgebung:")
print(trefferquote)
print()

gesunde = fehler[fehler[TARGET] == 0].groupby("env").size().rename("gesunde_laeufe")
alarme = (
    fehler[(fehler[TARGET] == 0) & (fehler["vorhersage"] == 1)]
    .groupby("env")
    .size()
    .rename("fehlalarme")
)
fehlalarmquote = gesunde.to_frame().join(alarme).fillna(0).astype(int)
fehlalarmquote["fehlalarmquote"] = (
    fehlalarmquote["fehlalarme"] / fehlalarmquote["gesunde_laeufe"]
).round(3)
print("Fehlalarmquote je Umgebung:")
print(fehlalarmquote)

Negativ-Methoden in der Validierung:
['pos_010_post_sign_in_user', 'pos_011_post_create_user', 'pos_032_login', 'pos_035_list_users', 'pos_079_manager_authenticate']

Übersehene Mutationen (FN): 3
                target_method      env     score
263        pos_035_list_users  extreme  0.136239
118  pos_011_post_create_user  extreme  0.082862
117  pos_011_post_create_user     high  0.079128

Erkennungsquote je Umgebung:
         mutationen  erkannt  trefferquote
env                                       
extreme           5        3           0.6
high              5        4           0.8
low               5        5           1.0
medium            5        5           1.0
prod              5        5           1.0

Fehlalarmquote je Umgebung:
         gesunde_laeufe  fehlalarme  fehlalarmquote
env                                                
extreme              21          11           0.524
high                 21          13           0.619
low                  21          10    

7. Ehrliche Auswertung und Label-Qualität

Die 0,607 aus der Validierung ist nicht belastbar, weil dort zufällig nur die fünf starken Methoden gelandet sind. Also rechne ich jetzt über alle 510 Zeilen: fünf Gruppierungen (GroupKFold über pair_id), jedes Modell sieht vier Fünftel der Methoden und bewertet das übrige Fünftel. So bekommt jede Zeile eine Vorhersage von einem Modell, das sie nicht kennt, und ich sehe zusätzlich die Streuung zwischen den Gruppen.

Danach derselbe Durchlauf ohne die fünf Grenzfälle (029, 041, 046, 056, 063). Diese Methoden waren in den Rohdaten nicht vom gesunden Verhalten zu unterscheiden - das ist keine Modellschwäche, sondern eine Grenze der Messung. Ich lasse sie deshalb aus den Negativ-Beispielen weg und stelle beide Varianten gegenüber. Das Ausschlusskriterium ist der Kampagnen-Befund von vor der Modellierung und nicht der Modellfehler, sonst wäre es Cherry-Picking.

Wichtig: Preprocessing und Modell stecken jetzt in einer Pipeline und werden pro Gruppierung neu angepasst. Nur so stammen Median, Skalierung und Kategorien ausschließlich aus den Trainingszeilen der jeweiligen Gruppierung.

In [145]:
from sklearn.base import clone
from sklearn.metrics import precision_recall_curve

GRENZFAELLE = ["_029_", "_041_", "_046_", "_056_", "_063_"]

spalten = LOG_FEATURES + PLAIN_FEATURES + CAT_FEATURES


def basis_modell():
    return Pipeline(
        steps=[
            ("prep", clone(preprocessor)),
            ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        ]
    )


def auswerten(daten, beschriftung, modell_bauen=basis_modell):
    X = daten[spalten]
    y = daten[TARGET].to_numpy()
    scores = np.zeros(len(daten))
    ap_pro_fold = []

    for train_teil, test_teil in GroupKFold(n_splits=5).split(
        X, y, groups=daten["pair_id"]
    ):
        modell = modell_bauen()
        modell.fit(X.iloc[train_teil], y[train_teil])
        teil_scores = modell.predict_proba(X.iloc[test_teil])[:, 1]
        scores[test_teil] = teil_scores
        ap_pro_fold.append(average_precision_score(y[test_teil], teil_scores))

    precision, recall, schwellen = precision_recall_curve(y, scores)
    geeignet = recall[:-1] >= 0.8
    schwelle = schwellen[geeignet].max()
    vorhersage = (scores >= schwelle).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, vorhersage).ravel()

    print(beschriftung)
    print("Zeilen:", len(daten), "| Negative:", int(y.sum()), "| Zufalls-AP:", round(y.mean(), 3))
    print(
        "AP je Gruppe:",
        [round(wert, 3) for wert in ap_pro_fold],
        "| Mittel:",
        round(np.mean(ap_pro_fold), 3),
        "+/-",
        round(np.std(ap_pro_fold), 3),
    )
    print("AP gepoolt (Out-of-Fold):", round(average_precision_score(y, scores), 3))
    print(
        "bei Schwelle",
        round(schwelle, 3),
        "-> Precision:",
        round(precision_score(y, vorhersage, zero_division=0), 3),
        "| Recall:",
        round(recall_score(y, vorhersage, zero_division=0), 3),
        "| F1:",
        round(f1_score(y, vorhersage, zero_division=0), 3),
    )
    print("TP/FP/FN/TN:", tp, fp, fn, tn)
    print()

    return {
        "Variante": beschriftung,
        "AP gepoolt": round(average_precision_score(y, scores), 3),
        "AP Mittel": round(np.mean(ap_pro_fold), 3),
        "AP Streuung": round(np.std(ap_pro_fold), 3),
    }


ohne_grenzfaelle = train_val[
    ~(
        (train_val[TARGET] == 1)
        & train_val["target_method"].str.contains("|".join(GRENZFAELLE))
    )
]

ergebnis_df = pd.DataFrame(
    [
        auswerten(train_val, "Basis, alle 21 Negativ-Methoden"),
        auswerten(ohne_grenzfaelle, "Basis, ohne die fünf Grenzfälle"),
    ]
).set_index("Variante")

print(ergebnis_df)

Basis, alle 21 Negativ-Methoden
Zeilen: 510 | Negative: 105 | Zufalls-AP: 0.206
AP je Gruppe: [0.453, 0.432, 0.444, 0.716, 0.52] | Mittel: 0.513 +/- 0.106
AP gepoolt (Out-of-Fold): 0.425
bei Schwelle 0.151 -> Precision: 0.313 | Recall: 0.8 | F1: 0.45
TP/FP/FN/TN: 84 184 21 221

Basis, ohne die fünf Grenzfälle
Zeilen: 485 | Negative: 80 | Zufalls-AP: 0.165
AP je Gruppe: [0.441, 0.479, 0.436, 0.72, 0.52] | Mittel: 0.519 +/- 0.105
AP gepoolt (Out-of-Fold): 0.405
bei Schwelle 0.099 -> Precision: 0.257 | Recall: 0.8 | F1: 0.389
TP/FP/FN/TN: 64 185 16 220

                                 AP gepoolt  AP Mittel  AP Streuung
Variante                                                           
Basis, alle 21 Negativ-Methoden       0.425      0.513        0.106
Basis, ohne die fünf Grenzfälle       0.405      0.519        0.105


8. Unterschiedliche Steigungen je Laststufe

Die Fehleranalyse hat gezeigt, dass das Modell bei high und extreme schlechter erkennt. Der Grund steckt im Aufbau des Modells: Mit den Umgebungs-Spalten kann es nur das Niveau je Laststufe verschieben, nicht die Steigung der KPI. Ein Effekt, der bei geringer Last kaum messbar ist und bei hoher Last deutlich ausfällt - genau das war das Ergebnis der Kampagne - lässt sich damit nicht abbilden.

Deshalb multipliziere ich hier jede KPI mit jeder Umgebungs-Spalte. Das Modell bekommt dadurch für jede Kombination einen eigenen Koeffizienten und kann pro Laststufe unterschiedlich stark auf Latenz oder Durchsatz reagieren. Die Grundmerkmale bleiben erhalten, die sechzehn Produkte kommen dazu.

Beide Konfigurationen - mit und ohne Grenzfälle - laufen durch dieselbe Auswertung wie in der Zelle davor, damit die Zahlen direkt vergleichbar sind.

In [146]:
from sklearn.preprocessing import FunctionTransformer

ANZAHL_KPI = len(LOG_FEATURES) + len(PLAIN_FEATURES)


def mit_interaktionen(X):
    kpi = X[:, :ANZAHL_KPI]
    umgebung = X[:, ANZAHL_KPI:]
    produkte = (kpi[:, :, None] * umgebung[:, None, :]).reshape(X.shape[0], -1)
    return np.hstack([X, produkte])


def interaktions_modell():
    return Pipeline(
        steps=[
            ("prep", clone(preprocessor)),
            ("inter", FunctionTransformer(mit_interaktionen)),
            ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        ]
    )


vergleich_df = pd.DataFrame(
    [
        auswerten(train_val, "Basis, alle 21 Negativ-Methoden"),
        auswerten(ohne_grenzfaelle, "Basis, ohne die fünf Grenzfälle"),
        auswerten(
            train_val,
            "Mit Interaktionen, alle 21 Negativ-Methoden",
            modell_bauen=interaktions_modell,
        ),
        auswerten(
            ohne_grenzfaelle,
            "Mit Interaktionen, ohne die fünf Grenzfälle",
            modell_bauen=interaktions_modell,
        ),
    ]
).set_index("Variante")

print(vergleich_df)

Basis, alle 21 Negativ-Methoden
Zeilen: 510 | Negative: 105 | Zufalls-AP: 0.206
AP je Gruppe: [0.453, 0.432, 0.444, 0.716, 0.52] | Mittel: 0.513 +/- 0.106
AP gepoolt (Out-of-Fold): 0.425
bei Schwelle 0.151 -> Precision: 0.313 | Recall: 0.8 | F1: 0.45
TP/FP/FN/TN: 84 184 21 221

Basis, ohne die fünf Grenzfälle
Zeilen: 485 | Negative: 80 | Zufalls-AP: 0.165
AP je Gruppe: [0.441, 0.479, 0.436, 0.72, 0.52] | Mittel: 0.519 +/- 0.105
AP gepoolt (Out-of-Fold): 0.405
bei Schwelle 0.099 -> Precision: 0.257 | Recall: 0.8 | F1: 0.389
TP/FP/FN/TN: 64 185 16 220

Mit Interaktionen, alle 21 Negativ-Methoden
Zeilen: 510 | Negative: 105 | Zufalls-AP: 0.206
AP je Gruppe: [0.455, 0.408, 0.297, 0.69, 0.509] | Mittel: 0.472 +/- 0.129
AP gepoolt (Out-of-Fold): 0.364
bei Schwelle 0.148 -> Precision: 0.308 | Recall: 0.8 | F1: 0.444
TP/FP/FN/TN: 84 189 21 216

Mit Interaktionen, ohne die fünf Grenzfälle
Zeilen: 485 | Negative: 80 | Zufalls-AP: 0.165
AP je Gruppe: [0.439, 0.432, 0.287, 0.668, 0.536] | Mittel: 

9. Zweites Modell: LightGBM

Die Logistische Regression zieht eine gerade Trennlinie zwischen gesund und mutiert. Bei Latenzwerten ist das eine starke Vereinfachung: Ein Anstieg von 300 auf 600 Millisekunden wirkt dort anders als einer von 2000 auf 4000. Die Interaktionsmerkmale sollten das auffangen und haben es schlechter gemacht, weil aus 8 Merkmalen auf 380 Zeilen gleich 24 wurden.

Deshalb probiere ich jetzt ein Modell, das Schwellen und Krümmungen selbst lernt: LightGBM mit bewusst kleinen Bäumen. Die Bäume bleiben flach (höchstens sieben Blätter), jedes Blatt braucht mindestens zwanzig Zeilen, und die Regularisierung ist aktiv. Ohne diese Bremsen merkt sich der Baumbau bei dieser Datenmenge einzelne Zeilen und liefert auf neuen Daten nichts mehr.

Das Preprocessing bleibt unverändert im Modell, obwohl Bäume keine Skalierung brauchen. So unterscheiden sich die beiden Läufe nur im Klassifikator, und ich vergleiche tatsächlich zwei Modelle und nicht zwei Vorverarbeitungen.

Wichtig: P12 und P13 werden hier nicht angefasst. Alle Zahlen kommen weiterhin aus der gruppierten Kreuzvalidierung über P01 bis P11, die unabhängige Prüfung bleibt für den Schluss reserviert.

Untersuchungsschritt: Die hier gewählte Merkmalsmenge (vier KPI plus Umgebung) ist nicht der Endstand. Das endgültige Modell steht in Abschnitt 11.

In [147]:
from lightgbm import LGBMClassifier


def lightgbm_modell():
    return Pipeline(
        steps=[
            ("prep", clone(preprocessor)),
            (
                "clf",
                LGBMClassifier(
                    n_estimators=300,
                    learning_rate=0.05,
                    num_leaves=7,
                    min_child_samples=20,
                    subsample=0.8,
                    subsample_freq=1,
                    colsample_bytree=0.8,
                    reg_lambda=1.0,
                    random_state=RANDOM_STATE,
                    verbose=-1,
                ),
            ),
        ]
    )


vergleich_lgbm = pd.DataFrame(
    [
        auswerten(train_val, "LogReg, alle 21 Methoden"),
        auswerten(ohne_grenzfaelle, "LogReg, ohne die fünf Grenzfälle"),
        auswerten(train_val, "LightGBM, alle 21 Methoden", modell_bauen=lightgbm_modell),
        auswerten(
            ohne_grenzfaelle,
            "LightGBM, ohne die fünf Grenzfälle",
            modell_bauen=lightgbm_modell,
        ),
    ]
).set_index("Variante")

print(vergleich_lgbm)

wichtigkeits_modell = lightgbm_modell()
wichtigkeits_modell.fit(train_val[spalten], train_val[TARGET])
namen = wichtigkeits_modell.named_steps["prep"].get_feature_names_out()
bedeutung = wichtigkeits_modell.named_steps["clf"].feature_importances_

print()
print("Feature-Bedeutung LightGBM (alle Trainingszeilen):")
print(pd.Series(bedeutung, index=namen).sort_values(ascending=False))

LogReg, alle 21 Methoden
Zeilen: 510 | Negative: 105 | Zufalls-AP: 0.206
AP je Gruppe: [0.453, 0.432, 0.444, 0.716, 0.52] | Mittel: 0.513 +/- 0.106
AP gepoolt (Out-of-Fold): 0.425
bei Schwelle 0.151 -> Precision: 0.313 | Recall: 0.8 | F1: 0.45
TP/FP/FN/TN: 84 184 21 221

LogReg, ohne die fünf Grenzfälle
Zeilen: 485 | Negative: 80 | Zufalls-AP: 0.165
AP je Gruppe: [0.441, 0.479, 0.436, 0.72, 0.52] | Mittel: 0.519 +/- 0.105
AP gepoolt (Out-of-Fold): 0.405
bei Schwelle 0.099 -> Precision: 0.257 | Recall: 0.8 | F1: 0.389
TP/FP/FN/TN: 64 185 16 220

LightGBM, alle 21 Methoden
Zeilen: 510 | Negative: 105 | Zufalls-AP: 0.206
AP je Gruppe: [0.623, 0.525, 0.606, 0.902, 0.448] | Mittel: 0.621 +/- 0.154
AP gepoolt (Out-of-Fold): 0.623
bei Schwelle 0.064 -> Precision: 0.356 | Recall: 0.8 | F1: 0.493
TP/FP/FN/TN: 84 152 21 253

LightGBM, ohne die fünf Grenzfälle
Zeilen: 485 | Negative: 80 | Zufalls-AP: 0.165
AP je Gruppe: [0.608, 0.471, 0.674, 0.853, 0.691] | Mittel: 0.659 +/- 0.124
AP gepoolt (Out

10. SHAP: welches Merkmal zieht in welche Richtung?

Die Feature-Bedeutung von LightGBM sagt mir nur, welches Merkmal oft zum Einsatz kommt - nicht, ab welchem Wert es die Bewertung kippt. Genau das brauche ich aber für die Bewertungsmatrix in Kapitel 4.5, sonst bleiben die Schwellen geraten.

SHAP zerlegt jede einzelne Vorhersage in Beiträge pro Merkmal. Positiv heißt: treibt die Bewertung in Richtung "mutiert", negativ heißt: spricht für gesund. Weil das Modell auf den logarithmierten und standardisierten Werten rechnet, sortiere ich die Beiträge anschließend wieder nach den Originalwerten - so stehen in der Ausgabe Millisekunden und Anfragen pro Sekunde, nicht Standardabweichungen.

Zwei Auswertungen: eine Rangfolge nach mittlerem Betrag der Beiträge, und je Merkmal fünf gleich große Wertebereiche mit dem durchschnittlichen Beitrag. Diese fünf Bereiche zeigen, wo das Vorzeichen umschlägt - und damit den ungefähren Schwellenwert.

Wichtig: Das Modell wird hier auf allen Trainingszeilen angelernt und nur erklärt, nicht bewertet. Die Zahlen aus der Kreuzvalidierung bleiben unberührt.

In [148]:
import shap

daten_shap = ohne_grenzfaelle.copy()
modell_shap = lightgbm_modell()
modell_shap.fit(daten_shap[spalten], daten_shap[TARGET])

vorbereitet = modell_shap.named_steps["prep"].transform(daten_shap[spalten])
namen = modell_shap.named_steps["prep"].get_feature_names_out()

erklaerer = shap.TreeExplainer(modell_shap.named_steps["clf"])
werte = erklaerer.shap_values(vorbereitet)
if isinstance(werte, list):
    werte = werte[1]
elif getattr(werte, "ndim", 2) == 3:
    werte = werte[:, :, 1]

shap_df = pd.DataFrame(werte, columns=namen)
original = daten_shap[list(namen[:4])].reset_index(drop=True)

print("Mittlerer Betrag der SHAP-Beiträge:")
print(shap_df.abs().mean().sort_values(ascending=False).round(3))
print()

for merkmal in ["requests_per_sec", "p95_latency_ms", "avg_latency_ms"]:
    bereiche = pd.qcut(original[merkmal], 5, duplicates="drop")
    uebersicht = pd.DataFrame(
        {
            "wert_von": original[merkmal].groupby(bereiche, observed=True).min().round(1),
            "wert_bis": original[merkmal].groupby(bereiche, observed=True).max().round(1),
            "shap_mittel": shap_df[merkmal].groupby(bereiche, observed=True).mean().round(3),
        }
    )
    print(merkmal)
    print(uebersicht)
    print()

Mittlerer Betrag der SHAP-Beiträge:
requests_per_sec      1.472
p95_latency_ms        0.770
avg_latency_ms        0.703
error_rate_percent    0.701
env_extreme           0.062
env_high              0.044
env_prod              0.040
env_medium            0.021
dtype: float64

requests_per_sec
                  wert_von  wert_bis  shap_mittel
requests_per_sec                                 
(0.339, 3.35]          0.3       3.4        1.796
(3.35, 6.96]           3.4       7.0       -1.576
(6.96, 40.654]         7.0      40.6        1.122
(40.654, 85.708]      40.7      85.6        0.055
(85.708, 234.94]      86.0     234.9       -1.384

p95_latency_ms
                    wert_von  wert_bis  shap_mittel
p95_latency_ms                                     
(6.999, 240.0]             7       240       -0.211
(240.0, 500.0]           250       500       -0.968
(500.0, 3300.0]          510      3300       -0.246
(3300.0, 16000.0]       3400     16000        0.450
(16000.0, 79000.0]     17000 

/Users/svenniederlohner/projects/Bachelorthesis_KI_gestuetztes_deployment/.venv/lib/python3.13/site-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


11. Endgültiges Modell

Die Untersuchung ist abgeschlossen, hier wird das Modell festgelegt. Es nutzt drei Merkmale: mittlere Latenz, p95-Latenz und Durchsatz. Keine Umgebungsmerkmale, weil sie messbar nichts beitragen (p = 0,99). Keine Fehlerquote, weil sie in diesen Daten kein Mutationssignal ist, sondern ein Lastsignal: Sie steigt in den gesunden Referenzläufen unter hoher Last, trennt die Klassen in der Randverteilung nicht und wirkt im Modell in umgekehrter Richtung als fachlich erwartet. Der Preis des Verzichts ist beziffert und klein.

Zur Kontrolle läuft die Variante mit Fehlerquote daneben, damit der Unterschied sichtbar bleibt und nicht der Eindruck entsteht, ein wirksames Merkmal sei ohne Grund entfernt worden.

Bewertet wird über zwei Wege. Zehn gruppierte Aufteilungen liefern die mittlere AP mit ihrer Streuung, weil eine einzelne Aufteilung bei dieser Datenmenge um den Faktor sieben schwanken kann. Die fünffache gruppierte Kreuzvalidierung liefert zusätzlich Precision, Recall und die Verwirrungsmatrix bei der Entscheidungsgrenze, die aus den Trainingsdaten für eine Trefferquote von 80 Prozent bestimmt wird - also ohne Blick auf die Prüfzeilen.

Gerechnet wird auf P01 bis P11 ohne die fünf Grenzfälle (029, 041, 046, 056, 063). Diese Methoden waren in den Rohdaten nicht vom gesunden Verhalten zu unterscheiden; dass sie ausgeschlossen sind, ist der Kampagnen-Befund von vor der Modellierung, nicht das Ergebnis der Modellfehler.

Alle Zellen davor sind Untersuchungsschritte: Sie zeigen, wie es zu dieser Wahl kam (lineares Modell, Interaktionsmerkmale, Merkmalsprüfung, Modellvergleich). Gültig für die Arbeit ist das Modell aus diesem Abschnitt.

In [149]:
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)

FEATURES_FINAL = ["avg_latency_ms", "p95_latency_ms", "requests_per_sec"]
FEATURES_REFERENZ = FEATURES_FINAL + ["error_rate_percent"]


def lgbm_klassifikator():
    return LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=7,
        min_child_samples=20,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        verbose=-1,
    )


def vorbereitung(numerische, mit_fehlerquote):
    schritte = [
        (
            "log",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numerische,
        )
    ]
    if mit_fehlerquote:
        schritte.append(
            (
                "plain",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                ["error_rate_percent"],
            )
        )
    return ColumnTransformer(
        transformers=schritte, sparse_threshold=0.0, verbose_feature_names_out=False
    )


def modell_bauen(numerische, mit_fehlerquote):
    return Pipeline(
        steps=[
            ("prep", vorbereitung(numerische, mit_fehlerquote)),
            ("clf", lgbm_klassifikator()),
        ]
    )


def pruefe(daten, features, mit_fehlerquote, beschriftung):
    X = daten[features]
    y = daten[TARGET].to_numpy()
    gruppen = daten["pair_id"].to_numpy()

    werte = []
    for seed in range(1, 11):
        train_teil, test_teil = next(
            GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed).split(
                X, y, groups=gruppen
            )
        )
        modell = modell_bauen(features, mit_fehlerquote)
        modell.fit(X.iloc[train_teil], y[train_teil])
        scores = modell.predict_proba(X.iloc[test_teil])[:, 1]
        werte.append(average_precision_score(y[test_teil], scores))

    scores_kreuz = np.zeros(len(daten))
    for train_teil, test_teil in GroupKFold(n_splits=5).split(X, y, groups=gruppen):
        modell = modell_bauen(features, mit_fehlerquote)
        modell.fit(X.iloc[train_teil], y[train_teil])
        scores_kreuz[test_teil] = modell.predict_proba(X.iloc[test_teil])[:, 1]

    precision, recall, schwellen = precision_recall_curve(y, scores_kreuz)
    schwelle = schwellen[recall[:-1] >= 0.8].max()
    vorhersage = (scores_kreuz >= schwelle).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, vorhersage).ravel()

    print(beschriftung)
    print("  Merkmale:", features)
    print(
        "  AP aus zehn Aufteilungen:",
        round(np.mean(werte), 3),
        "+/-",
        round(np.std(werte), 3),
    )
    print("  AP aus fünffacher Kreuzvalidierung:", round(average_precision_score(y, scores_kreuz), 3))
    print("  Zufallswert:", round(y.mean(), 3), "| Entscheidungsgrenze:", round(schwelle, 3))
    print(
        "  Precision:",
        round(precision_score(y, vorhersage, zero_division=0), 3),
        "| Recall:",
        round(recall_score(y, vorhersage, zero_division=0), 3),
        "| F1:",
        round(f1_score(y, vorhersage, zero_division=0), 3),
    )
    print("  Verwirrungsmatrix  TN:", tn, "FP:", fp, "FN:", fn, "TP:", tp)
    print()

    return {
        "Variante": beschriftung,
        "AP 10 Aufteilungen": round(np.mean(werte), 3),
        "AP Streuung": round(np.std(werte), 3),
        "AP Kreuzvalidierung": round(average_precision_score(y, scores_kreuz), 3),
        "Precision": round(precision_score(y, vorhersage, zero_division=0), 3),
        "Recall": round(recall_score(y, vorhersage, zero_division=0), 3),
        "F1": round(f1_score(y, vorhersage, zero_division=0), 3),
        "FP": fp,
        "FN": fn,
    }


abschluss_df = pd.DataFrame(
    [
        pruefe(ohne_grenzfaelle, FEATURES_FINAL, False, "Endmodell (3 Merkmale)"),
        pruefe(
            ohne_grenzfaelle,
            FEATURES_FINAL + ["error_rate_percent"],
            True,
            "Referenz (mit Fehlerquote)",
        ),
    ]
).set_index("Variante")

print(abschluss_df)

Endmodell (3 Merkmale)
  Merkmale: ['avg_latency_ms', 'p95_latency_ms', 'requests_per_sec']
  AP aus zehn Aufteilungen: 0.503 +/- 0.172
  AP aus fünffacher Kreuzvalidierung: 0.601
  Zufallswert: 0.165 | Entscheidungsgrenze: 0.027
  Precision: 0.278 | Recall: 0.8 | F1: 0.413
  Verwirrungsmatrix  TN: 239 FP: 166 FN: 16 TP: 64

Referenz (mit Fehlerquote)
  Merkmale: ['avg_latency_ms', 'p95_latency_ms', 'requests_per_sec', 'error_rate_percent']
  AP aus zehn Aufteilungen: 0.522 +/- 0.168
  AP aus fünffacher Kreuzvalidierung: 0.651
  Zufallswert: 0.165 | Entscheidungsgrenze: 0.034
  Precision: 0.308 | Recall: 0.8 | F1: 0.444
  Verwirrungsmatrix  TN: 261 FP: 144 FN: 16 TP: 64

                            AP 10 Aufteilungen  AP Streuung  \
Variante                                                      
Endmodell (3 Merkmale)                   0.503        0.172   
Referenz (mit Fehlerquote)               0.522        0.168   

                            AP Kreuzvalidierung  Precision  Recall 

12. Datensatz für die Code-Vorhersage

Hier entsteht der Trainingsdatensatz für den Teil, der die eigentliche Vorhersage macht: Aus dem Code soll abgeleitet werden, wie sich die Messwerte verhalten werden - bevor gemessen wurde.

Das Etikett kommt nicht aus den Modellfehlern, sondern aus der Bewertungsmatrix. Für jede Zeile wird über den Prototypen der Punktwert berechnet: Abweichung zur Referenz derselben Methode und Umgebungsstufe, Bandzuordnung, gewichtete Summe. Daraus entstehen zwei Ziele. Das erste ist die Einstufung in stabil, Grenzfall oder instabil. Das zweite ist das Hauptkriterium - also die Frage, welcher der drei Werte die Bewertung nach unten gezogen hat; dieses Ziel ist die interessantere Aussage, weil der Agent später sagen soll, welche Kennzahl leiden wird.

Genutzt wird der Analysesatz: P01 bis P11, ohne das ausgeschlossene Projekt P03 und ohne die fünf nicht trennscharfen Mutationen. Das sind 485 Zeilen, davon 405 gesund und 80 mutiert, alle mit Snippet-Code.

Eine Einschränkung, die in die Arbeit gehört: Gesunde Läufe bekommen den Punktwert 100, weil sie mit ihrer eigenen Referenz verglichen werden - sie können per Konstruktion keine Abweichung haben. Das Etikett ist dadurch deutlich schiefer als das ursprüngliche KPI-Label: Nur die Mutationen verteilen sich auf die drei Einstufungen, und ein Teil von ihnen erhält sogar stabil, weil die Matrix ihre Wirkung nicht erkennt. Genau diese Zeilen sind für das Code-Modell nicht lernbar und müssen bei der Bewertung als Grenze benannt werden.

In [150]:
import json
import sys

sys.path.insert(0, str(ROOT / "prototype"))
from scoring import bewerte, lade_hilfsdaten

referenz, notgrenzen = lade_hilfsdaten()

GRENZFAELLE_NUMMERN = {"029", "041", "046", "056", "063"}
KURZNAMEN = {
    "Durchsatz gegenüber Referenz": "durchsatz",
    "p95-Latenz gegenüber Referenz": "p95_latenz",
    "Mittlere Latenz gegenüber Referenz": "mittlere_latenz",
}

zeilen = [
    json.loads(zeile)
    for zeile in open(ROOT / "export_kpis" / "dataset.jsonl", encoding="utf-8")
]
analyse = [
    zeile
    for zeile in zeilen
    if zeile["split"] == "train"
    and not zeile["variant"].startswith("P03")
    and not (zeile["label"] == 1 and zeile["pos"] in GRENZFAELLE_NUMMERN)
]

saetze = []
for zeile in analyse:
    messung = {"target_method": zeile["method"], "env": zeile["env"], **zeile["metrics"]}
    antwort = bewerte(messung, referenz, notgrenzen)
    kriterien = {k["name"]: k for k in antwort["kriterien"]}
    verluste = sorted(
        (
            (k["gewicht"] * (100 - k["punkte"]), KURZNAMEN[k["name"]])
            for k in antwort["kriterien"]
        ),
        reverse=True,
    )
    saetze.append(
        {
            "variante": zeile["variant"],
            "methode": zeile["method"],
            "nummer": zeile["pos"],
            "umgebung": zeile["env"],
            "label_kpi": zeile["label"],
            "score_prozent": antwort["score_prozent"],
            "ziel_einstufung": antwort["einstufung"],
            "ziel_auffaellig": int(antwort["score_prozent"] < 75),
            "ziel_kriterium": verluste[0][1] if verluste[0][0] > 0 else "keines",
            "durchsatz_punkte": kriterien["Durchsatz gegenüber Referenz"]["punkte"],
            "p95_punkte": kriterien["p95-Latenz gegenüber Referenz"]["punkte"],
            "avg_punkte": kriterien["Mittlere Latenz gegenüber Referenz"]["punkte"],
            "durchsatz_abweichung": kriterien["Durchsatz gegenüber Referenz"]["abweichung_prozent"],
            "p95_abweichung": kriterien["p95-Latenz gegenüber Referenz"]["abweichung_prozent"],
            "avg_abweichung": kriterien["Mittlere Latenz gegenüber Referenz"]["abweichung_prozent"],
            "snippet": zeile["snippet"],
        }
    )

daten = pd.DataFrame(saetze)

print("Zeilen im Analysesatz:", len(daten), "| mit Code:", int((daten["snippet"].str.len() > 0).sum()))
print()
print("Einstufung nach KPI-Label (0 = gesund, 1 = mutiert):")
print(pd.crosstab(daten["label_kpi"], daten["ziel_einstufung"]))
print()
print("Hauptkriterium bei den Mutationen:")
print(daten[daten["label_kpi"] == 1]["ziel_kriterium"].value_counts())
print()
print("Binäres Ziel (auffällig = Score unter 75):")
print(daten["ziel_auffaellig"].value_counts().rename({0: "unauffällig", 1: "auffällig"}))
print()
print(
    "Nicht erkannte Mutationen (Etikett unauffällig, obwohl mutiert):",
    int(((daten["label_kpi"] == 1) & (daten["ziel_auffaellig"] == 0)).sum()),
)
print("Mehrheitslinie für das binäre Ziel:", round(daten["ziel_auffaellig"].mean(), 3))
print()
print("Snippet-Länge (Zeichen):")
print(daten["snippet"].str.len().describe().round(0))

ziel_datei = ROOT / "export_kpis" / "dataset_mit_labels.csv"
daten.to_csv(ziel_datei, index=False)
print()
print("Gespeichert:", ziel_datei)

Zeilen im Analysesatz: 485 | mit Code: 485

Einstufung nach KPI-Label (0 = gesund, 1 = mutiert):
ziel_einstufung  Grenzfall  instabil  stabil
label_kpi                                   
0                        0         0     405
1                       28        34      18

Hauptkriterium bei den Mutationen:
ziel_kriterium
durchsatz          68
keines              5
p95_latenz          5
mittlere_latenz     2
Name: count, dtype: int64

Binäres Ziel (auffällig = Score unter 75):
ziel_auffaellig
unauffällig    423
auffällig       62
Name: count, dtype: int64

Nicht erkannte Mutationen (Etikett unauffällig, obwohl mutiert): 18
Mehrheitslinie für das binäre Ziel: 0.128

Snippet-Länge (Zeichen):
count     485.0
mean     1255.0
std       662.0
min       302.0
25%       772.0
50%      1103.0
75%      1491.0
max      3433.0
Name: snippet, dtype: float64

Gespeichert: /Users/svenniederlohner/projects/Bachelorthesis_KI_gestuetztes_deployment/export_kpis/dataset_mit_labels.csv


13. Variante B: nur der Code

Jetzt kommt die eigentliche Frage der Arbeit. Bisher wurde aus Messwerten gelernt - das Modell sieht die KPI und sagt, ob ein Lauf gesund oder mutiert ist. Das nützt beim Pull Request nichts, weil es dort noch keine Messung gibt.

Variante B bekommt deshalb ausschließlich den Code zu sehen: das Snippet der geänderten Methode, sonst nichts. Vorhergesagt wird das Etikett aus der Matrix - also die Frage, ob diese Änderung auffällig werden wird.

Als Merkmale dient der Snippet-Text über Zeichenfolgen von drei bis fünf Zeichen. Quellcode variiert stark in Einrückung, Aufrufketten und Bezeichnern, weshalb Zeichenfolgen robuster sind als Wörter.

Gerechnet wird mit drei Gruppierungen, und das ist hier der entscheidende Punkt. Erstens über die Methodennummer, zweitens über das Projekt - dabei bleiben Original und zugehörige Mutation desselben Projekts zusammen, sonst lernt das Modell den Projektstil statt das Fehlermuster -, drittens zum Vergleich über die Variantenkennung, die diese Einheit auseinanderreißt. Angegeben werden PR-AUC, Zufallswert, Präzision, Trefferquote und F1 je Konfiguration, dazu die tragenden Zeichenfolgen.

Zum Vergleich steht die KPI-Baseline aus Variante A mit 0,50 bis 0,62 bereit.

In [151]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline

daten = pd.read_csv(ROOT / "export_kpis" / "dataset_mit_labels.csv", dtype={"nummer": str})
daten["projekt"] = daten["variante"].str.replace("_neg", "", regex=False)
ZIEL = "ziel_auffaellig"
y = daten[ZIEL].to_numpy()
X = daten["snippet"]


def baue_code_modell():
    return Pipeline(
        steps=[
            ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2)),
            ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
        ]
    )


def kennzahlen(gruppe, verfahren):
    if verfahren == "faltungen":
        teiler = list(GroupKFold(n_splits=5).split(X, y, groups=daten[gruppe].to_numpy()))
    else:
        teiler = [
            next(
                GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed).split(
                    X, y, groups=daten[gruppe].to_numpy()
                )
            )
            for seed in range(1, 11)
        ]
    zeilen = []
    for train_teil, test_teil in teiler:
        if y[test_teil].sum() == 0:
            continue
        modell = baue_code_modell()
        modell.fit(X.iloc[train_teil], y[train_teil])
        scores = modell.predict_proba(X.iloc[test_teil])[:, 1]
        precision, recall, schwellen = precision_recall_curve(y[test_teil], scores)
        geeignet = recall[:-1] >= 0.8
        schwelle = schwellen[geeignet].max() if geeignet.any() else 1.0
        vorhersage = (scores >= schwelle).astype(int)
        zeilen.append(
            {
                "PR-AUC": average_precision_score(y[test_teil], scores),
                "Zufall": y[test_teil].mean(),
                "Praezision": precision_score(y[test_teil], vorhersage, zero_division=0),
                "Trefferquote": recall_score(y[test_teil], vorhersage, zero_division=0),
                "F1": f1_score(y[test_teil], vorhersage, zero_division=0),
            }
        )
    return pd.DataFrame(zeilen)


print("Ziel:", ZIEL, "| auffällig:", int(y.sum()), "von", len(y), "| Prävalenz:", round(y.mean(), 3))
print("Gruppen: Methode", daten["nummer"].nunique(), "| Projekt", daten["projekt"].nunique(), "| Variante", daten["variante"].nunique())
print()

for gruppe, verfahren, name in [
    ("nummer", "aufteilungen", "Gruppe Methode, 10 Aufteilungen"),
    ("projekt", "aufteilungen", "Gruppe Projekt projekt-rein, 10 Aufteilungen"),
    ("projekt", "faltungen", "Gruppe Projekt projekt-rein, 5 Faltungen"),
    ("variante", "faltungen", "Gruppe Variante, 5 Faltungen"),
]:
    tabelle = kennzahlen(gruppe, verfahren)
    print(f"{name}")
    print(f"   PR-AUC {round(tabelle['PR-AUC'].mean(), 3)} +/- {round(tabelle['PR-AUC'].std(), 3)}"
          f" | Zufallswert {round(tabelle['Zufall'].mean(), 3)}"
          f" | Präzision {round(tabelle['Praezision'].mean(), 3)}"
          f" | Trefferquote {round(tabelle['Trefferquote'].mean(), 3)}"
          f" | F1 {round(tabelle['F1'].mean(), 3)}")
    print(f"   Einzelwerte PR-AUC: {[round(wert, 3) for wert in tabelle['PR-AUC']]}")
    print()

gesamt = baue_code_modell()
gesamt.fit(X, y)
merkmalnamen = gesamt.named_steps["tfidf"].get_feature_names_out()
koeffizienten = gesamt.named_steps["clf"].coef_[0]
sortiert = np.argsort(koeffizienten)
print("Stärkste Zeichenfolgen für 'auffällig':")
for i in sortiert[-15:][::-1]:
    print("   ", repr(merkmalnamen[i]), round(koeffizienten[i], 3))
print("Stärkste Zeichenfolgen für 'unauffällig':")
for i in sortiert[:10]:
    print("   ", repr(merkmalnamen[i]), round(koeffizienten[i], 3))

Ziel: ziel_auffaellig | auffällig: 62 von 485 | Prävalenz: 0.128
Gruppen: Methode 81 | Projekt 10 | Variante 18

Gruppe Methode, 10 Aufteilungen
   PR-AUC 0.768 +/- 0.171 | Zufallswert 0.106 | Präzision 0.611 | Trefferquote 0.934 | F1 0.702
   Einzelwerte PR-AUC: [0.635, 0.773, 0.777, 0.7, 1.0, 0.825, 1.0, 0.733, 0.82, 0.412]

Gruppe Projekt projekt-rein, 10 Aufteilungen
   PR-AUC 0.588 +/- 0.187 | Zufallswert 0.103 | Präzision 0.43 | Trefferquote 0.934 | F1 0.549
   Einzelwerte PR-AUC: [0.717, 0.25, 0.25, 0.665, 0.65, 0.689, 0.766, 0.717, 0.558, 0.622]

Gruppe Projekt projekt-rein, 5 Faltungen
   PR-AUC 0.797 +/- 0.153 | Zufallswert 0.122 | Präzision 0.663 | Trefferquote 0.978 | F1 0.76
   Einzelwerte PR-AUC: [0.89, 1.0, 0.689, 0.791, 0.617]

Gruppe Variante, 5 Faltungen
   PR-AUC 0.238 +/- 0.118 | Zufallswert 0.214 | Präzision 0.241 | Trefferquote 0.971 | F1 0.376
   Einzelwerte PR-AUC: [0.227, 0.361, 0.126]

Stärkste Zeichenfolgen für 'auffällig':
    ' al' 0.637
    'et ' 0.557
   

14. Variante D: billige Messung plus Code

Wenn schon eine Vorhersage ohne Messung möglich ist, dann interessiert die Frage, wie viel eine billige Messung zusätzlich bringt. Die Stufe low läuft schnell und kostet wenig. Variante D bekommt deshalb das Snippet, die Messwerte der Stufe low und die Zielumgebungsstufe - vorhergesagt wird das Etikett für die höhere Stufe.

Wichtig für die Sauberkeit: Als Merkmale dienen ausschließlich die Messwerte der Stufe low, nicht die der Zielstufe. Sonst würde die Antwort in der Eingabe stehen, weil das Etikett aus der Messung der Zielstufe berechnet wird.

Gerechnet werden drei Konfigurationen mit zwei Gruppierungen: nur die Low-Messung mit Zielumgebung, nur der Code und beides zusammen, jeweils über die Methode und über das Projekt.

Dazu kommt eine parameterfreie Regel als härteste Vergleichslinie: Ist der Lauf schon bei low auffällig, wird er es auch höher sein. Regeln ohne gelernte Parameter brauchen keine Aufteilung und können deshalb nicht durch eine günstige Gruppierung besser aussehen, als sie sind.

In [152]:
from sklearn.metrics import confusion_matrix

zeilen_d = [
    json.loads(zeile)
    for zeile in open(ROOT / "export_kpis" / "dataset.jsonl", encoding="utf-8")
]
metriken = pd.DataFrame(
    [
        {
            "variante": zeile["variant"],
            "nummer": str(zeile["pos"]).zfill(3),
            "umgebung": zeile["env"],
            "rps": zeile["metrics"]["requests_per_sec"],
            "avg": zeile["metrics"]["avg_latency_ms"],
            "p95": zeile["metrics"]["p95_latency_ms"],
        }
        for zeile in zeilen_d
        if zeile["snippet"]
    ]
)
niedrig = (
    metriken[metriken["umgebung"] == "low"][["variante", "nummer", "rps", "avg", "p95"]]
    .rename(columns={"rps": "rps_low", "avg": "avg_low", "p95": "p95_low"})
)

daten_d = pd.read_csv(ROOT / "export_kpis" / "dataset_mit_labels.csv", dtype={"nummer": str})
daten_d["projekt"] = daten_d["variante"].str.replace("_neg", "", regex=False)
daten_d = daten_d[
    daten_d["umgebung"].isin(["medium", "high", "extreme", "prod"])
].merge(niedrig, on=["variante", "nummer"], how="left").reset_index(drop=True)

print("Zeilen:", len(daten_d), "| auffällig:", int(daten_d[ZIEL].sum()), "| Prävalenz:", round(daten_d[ZIEL].mean(), 3))
print()

y_d = daten_d[ZIEL].to_numpy()
KPI_LOW = ["rps_low", "avg_low", "p95_low"]


def modell_d(mit_text, mit_kpi, mit_umgebung):
    teile = []
    if mit_text:
        teile.append(("text", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2), "snippet"))
    if mit_kpi:
        teile.append(("kpi", Pipeline(steps=[("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")), ("skala", StandardScaler())]), KPI_LOW))
    if mit_umgebung:
        teile.append(("umgebung", OneHotEncoder(handle_unknown="ignore"), ["umgebung"]))
    return Pipeline(
        steps=[
            ("merkmale", ColumnTransformer(transformers=teile)),
            ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
        ]
    )


def pruefe_d(mit_text, mit_kpi, mit_umgebung, gruppe, verfahren, beschriftung):
    spalten = (["snippet"] if mit_text else []) + (KPI_LOW if mit_kpi else []) + (["umgebung"] if mit_umgebung else [])
    X = daten_d[spalten]
    if verfahren == "faltungen":
        teiler = list(GroupKFold(n_splits=5).split(X, y_d, groups=daten_d[gruppe].to_numpy()))
    else:
        teiler = [
            next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed).split(X, y_d, groups=daten_d[gruppe].to_numpy()))
            for seed in range(1, 11)
        ]
    werte = []
    for train_teil, test_teil in teiler:
        if y_d[test_teil].sum() == 0:
            continue
        modell = modell_d(mit_text, mit_kpi, mit_umgebung)
        modell.fit(X.iloc[train_teil], y_d[train_teil])
        werte.append(average_precision_score(y_d[test_teil], modell.predict_proba(X.iloc[test_teil])[:, 1]))
    print(f"{beschriftung:46s} {gruppe:8s} {verfahren:12s} PR-AUC {round(np.mean(werte), 3)} +/- {round(np.std(werte), 3)} (n={len(werte)})")


print("Projekt-reine Gruppierung (5 Faltungen):")
pruefe_d(False, True, True, "projekt", "faltungen", "Nur Low-Messung + Zielumgebung")
pruefe_d(True, False, False, "projekt", "faltungen", "Nur Code")
pruefe_d(True, True, True, "projekt", "faltungen", "Code + Low-Messung + Zielumgebung")
print()
print("Gruppierung über die Methode (10 Aufteilungen):")
pruefe_d(False, True, True, "nummer", "aufteilungen", "Nur Low-Messung + Zielumgebung")
pruefe_d(True, False, False, "nummer", "aufteilungen", "Nur Code")
pruefe_d(True, True, True, "nummer", "aufteilungen", "Code + Low-Messung + Zielumgebung")
print()

low_etikett = (
    pd.read_csv(ROOT / "export_kpis" / "dataset_mit_labels.csv", dtype={"nummer": str})
    .query("umgebung == 'low'")[["variante", "nummer", ZIEL]]
    .rename(columns={ZIEL: "low_auffaellig"})
)
regel = daten_d.merge(low_etikett, on=["variante", "nummer"], how="left")

print("Parameterfreie Regel: Etikett der Low-Stufe auf die höhere Stufe übertragen")
print(pd.crosstab(regel["low_auffaellig"], regel[ZIEL], rownames=["low auffällig"], colnames=["Ziel auffällig"]))
tn, fp, fn, tp = confusion_matrix(regel[ZIEL], regel["low_auffaellig"]).ravel()
print()
print("  Treffer:", tp, "| Fehlalarm:", fp, "| übersehen:", fn, "| korrekt freigegeben:", tn)
print("  Präzision:", round(tp / (tp + fp), 3), "| Trefferquote:", round(tp / (tp + fn), 3), "| F1:", round(2 * tp / (2 * tp + fp + fn), 3))

Zeilen: 388 | auffällig: 50 | Prävalenz: 0.129

Projekt-reine Gruppierung (5 Faltungen):
Nur Low-Messung + Zielumgebung                 projekt  faltungen    PR-AUC 0.561 +/- 0.304 (n=5)
Nur Code                                       projekt  faltungen    PR-AUC 0.794 +/- 0.142 (n=5)
Code + Low-Messung + Zielumgebung              projekt  faltungen    PR-AUC 0.704 +/- 0.226 (n=5)

Gruppierung über die Methode (10 Aufteilungen):
Nur Low-Messung + Zielumgebung                 nummer   aufteilungen PR-AUC 0.51 +/- 0.187 (n=10)
Nur Code                                       nummer   aufteilungen PR-AUC 0.758 +/- 0.184 (n=10)
Code + Low-Messung + Zielumgebung              nummer   aufteilungen PR-AUC 0.744 +/- 0.201 (n=10)

Parameterfreie Regel: Etikett der Low-Stufe auf die höhere Stufe übertragen
Ziel auffällig    0   1
low auffällig          
0               331   9
1                 7  41

  Treffer: 41 | Fehlalarm: 7 | übersehen: 9 | korrekt freigegeben: 331
  Präzision: 0.854 | Treffe